In [1]:
import re
import os
import cv2
import time

import numpy as np
import pandas as pd

from skimage import io
from skimage import feature
from skimage.measure import label, regionprops
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from scipy.stats import skew, kurtosis, entropy
from tqdm import tqdm

Sort and read

In [2]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def read_image_data(folder_path):
    image_names = sorted(os.listdir(folder_path), key=natural_sort_key)
    if not image_names:
        print("No images found in the directory.")
        return []
    elif len(image_names) > 2:
        print(f"The names of the first three images in the directory are: {image_names[0]}, {image_names[1]}, {image_names[2]}")
    else:
        print("Not enough images to display three names.")
    image_paths = [os.path.join(folder_path, name) for name in image_names]
    return image_paths

Basic feature

In [3]:
def extract_basic_features(image_paths):
    features = []
    for image_path in tqdm(image_paths):
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f"Failed to read image: {image_path}")
            continue
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, img_bin = cv2.threshold(img_gray, 128, 255, cv2.THRESH_BINARY)
        label_img = label(img_bin)
        props = regionprops(label_img)
        if props:
            largest_prop = max(props, key=lambda x: x.area)
            features.append([os.path.basename(image_path)] + [
                largest_prop.area,
                largest_prop.bbox[3] - largest_prop.bbox[1],
                largest_prop.bbox[2] - largest_prop.bbox[0],
                (largest_prop.bbox[3] - largest_prop.bbox[1]) / (largest_prop.bbox[2] - largest_prop.bbox[0]) if (largest_prop.bbox[2] - largest_prop.bbox[0]) != 0 else 0,
                largest_prop.major_axis_length,
                largest_prop.minor_axis_length,
                largest_prop.convex_area,
                cv2.arcLength(np.array(largest_prop.coords), closed=True),
                np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1], 0]),
                np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1], 1]),
                np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1], 2]),
                np.sqrt(np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1], 0])),
                np.sqrt(np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1], 1])),
                np.sqrt(np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1], 2])),
                np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1]]),
                np.std(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1]]),
                np.sum((img[largest_prop.coords[:, 0], largest_prop.coords[:, 1]] - np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1]])) ** 2),
                np.sum((img[largest_prop.coords[:, 0], largest_prop.coords[:, 1]] - np.mean(img[largest_prop.coords[:, 0], largest_prop.coords[:, 1]])) ** 3),
            ])
        else:
            features.append([os.path.basename(image_path)] + [np.nan] * 18 )

    df = pd.DataFrame(features, columns=['Name', 'area', 'length', 'width', 'length_width_ratio', 'major_axis_length',
                                         'minor_axis_length', 'convex_area', 'perimeter', 'r_mean', 'g_mean', 'b_mean',
                                         'rs', 'gs', 'bs', 'mean', 'std_dev', 'uniformity', 'third_moment',])
    return df

Advange basic features

In [4]:
def extract_advanced_features(image_paths):
    features = []
    for image_path in tqdm(image_paths):
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f"Failed to read image: {image_path}")
            continue

        # Preprocess the image
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, img_bin = cv2.threshold(img_gray, 128, 255, cv2.THRESH_BINARY)
        label_img = label(img_bin)
        props = regionprops(label_img)
        
        # Convert image to different color spaces
        img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        
        # Calculate additional features
        if props:
            largest_prop = max(props, key=lambda x: x.area)
            coords = largest_prop.coords
            region_pixels = img[coords[:, 0], coords[:, 1]]
            region_pixels_hsv = img_hsv[coords[:, 0], coords[:, 1]]

            r, g, b = region_pixels[:, 0], region_pixels[:, 1], region_pixels[:, 2]
            h, s, v = region_pixels_hsv[:, 0], region_pixels_hsv[:, 1], region_pixels_hsv[:, 2]
            normalized_r = r / (r + g + b + 0.01)
            normalized_g = g / (r + g + b + 0.01)
            normalized_b = b / (r + g + b + 0.01)
            brightness = 0.299 * r + 0.587 * g + 0.114 * b
            r_squared = r ** 2
            g_squared = g ** 2
            b_squared = b ** 2

            mean_hue = np.mean(h)
            mean_saturation = np.mean(s)
            hue_variance = np.var(h)
            saturation_variance = np.var(s)
            
            r_skewness = skew(r)
            g_skewness = skew(g)
            b_skewness = skew(b)
            r_kurtosis = kurtosis(r)
            g_kurtosis = kurtosis(g)
            b_kurtosis = kurtosis(b)
            
            color_entropy = entropy(np.histogram(region_pixels, bins=256)[0])
            # dominant_color = region_pixels[np.argmax(np.bincount(region_pixels[:, 0] * 256 * 256 + region_pixels[:, 1] * 256 + region_pixels[:, 2]))]
            
            feature_row = [os.path.basename(image_path)] + [
                # largest_prop.area,  # Commented out the area feature
                normalized_r.mean(), normalized_g.mean(), normalized_b.mean(),
                brightness.mean(), r_squared.mean(), g_squared.mean(), b_squared.mean(),
                mean_hue, mean_saturation, hue_variance, saturation_variance,
                r_skewness, g_skewness, b_skewness, r_kurtosis, g_kurtosis, b_kurtosis,
                color_entropy,
                # dominant_color
            ]
            features.append(feature_row)
        else:
            features.append([os.path.basename(image_path)] + [np.nan] * 18)  

    columns = ['Name', 'norm_r', 'norm_g', 'norm_b', 'brightness', 'r_squared', 'g_squared', 'b_squared',
               'mean_hue', 'mean_saturation', 'hue_variance', 'saturation_variance',
               'r_skewness', 'g_skewness', 'b_skewness', 'r_kurtosis', 'g_kurtosis', 'b_kurtosis',
               'color_entropy']
    df = pd.DataFrame(features, columns=columns)
    return df

Color Histogram single

In [ ]:
def color_histogram_1D(image_paths, bins=8):
    features = []
    for image_path in tqdm(image_paths):
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f"Failed to read {image_path}")
            continue
        
        b, g, r = cv2.split(img)
        r_hist = np.histogram(r, bins=bins, range=(0, 256), density=True)[0]
        g_hist = np.histogram(g, bins=bins, range=(0, 256), density=True)[0]
        b_hist = np.histogram(b, bins=bins, range=(0, 256), density=True)[0]

        feature_vector = [os.path.basename(image_path)] + list(r_hist) + list(g_hist) + list(b_hist)
        features.append(feature_vector)

    columns = ['Name'] + \
              [f'r_bin_{i}' for i in range(bins)] + \
              [f'g_bin_{i}' for i in range(bins)] + \
              [f'b_bin_{i}' for i in range(bins)]
    
    return pd.DataFrame(features, columns=columns)


def color_histogram_3D(image_paths, bins=8):
    features = []

    for image_path in tqdm(image_paths):
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f"Failed to read {image_path}")
            continue

        # Convert BGR to RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Compute 3D histogram (bins x bins x bins)
        hist = cv2.calcHist([img_rgb], [0, 1, 2], None, 
                            [bins, bins, bins], [0, 256, 0, 256, 0, 256])
        hist = cv2.normalize(hist, hist).flatten()

        feature_vector = [os.path.basename(image_path)] + hist.tolist()
        features.append(feature_vector)

    columns = ['Name'] + [f'rgb_bin_{i}' for i in range(bins ** 3)]
    return pd.DataFrame(features, columns=columns)


LBP feature

In [6]:
def extract_LBP_features(image_paths):
    data = []
    points = 8
    radius = 1
    for image_path in tqdm(image_paths):
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is not None:
            lbp = feature.local_binary_pattern(image, points, radius, method="uniform")
            (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, points + 3), range=(0, points + 2))
            hist = hist.astype("float")
            hist /= (hist.sum() + 1e-7)
            hist_series = pd.Series(hist, name=os.path.basename(image_path))
            data.append(hist_series)
        else:
            print(f"Failed to read {image_path}")
            data.append(pd.Series([np.nan]*10, name=os.path.basename(image_path)))
    lbp_df = pd.DataFrame(data).reset_index().rename(columns={"index": "Name"})
    return lbp_df

GIST feature

In [1]:
def extract_gist_features(image_paths, orientations=8, blocks=4):
    descriptors = []
    for path in tqdm(image_paths):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Failed to read image: {path}")
            descriptors.append([os.path.basename(path)] + [np.nan]*(orientations*blocks*blocks))
            continue
        height, width = img.shape
        cell_size = min(height, width) // blocks
        gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
        gradient_magnitude = np.sqrt(gx**2 + gy**2)
        gradient_orientation = np.arctan2(gy, gx) * (180 / np.pi) + 180
        gradient_orientation_bins = np.floor(gradient_orientation / (360 / orientations)).astype(int)
        descriptor = np.zeros(orientations * blocks * blocks)
        for i in range(blocks):
            for j in range(blocks):
                cell_hist = np.zeros(orientations)
                for ii in range(cell_size):
                    for jj in range(cell_size):
                        x = i * cell_size + ii
                        y = j * cell_size + jj
                        if x >= height or y >= width:
                            continue
                        bin_idx = gradient_orientation_bins[x, y] % orientations
                        cell_hist[bin_idx] += gradient_magnitude[x, y]
                descriptor[(i * blocks + j) * orientations:(i * blocks + j + 1) * orientations] = cell_hist
        descriptor /= (np.sum(descriptor) + 1e-7)
        descriptors.append([os.path.basename(path)] + descriptor.tolist())
    gist_df = pd.DataFrame(descriptors, columns=['Name'] + [f'GIST_{i}' for i in range(orientations * blocks * blocks)])
    return gist_df

GLCM feature

In [8]:
def GLCM_all(image_paths, distance=3):
    list_GLCM = []
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    for image_path in tqdm(image_paths):
        img = io.imread(image_path, as_gray=True, plugin='pil')
        if img is None:
            print(f"Failed to read image: {image_path}")
            continue
        img = img.astype(np.uint8)
        
        glcm = graycomatrix(img, [distance], angles, 256, symmetric=True, normed=True)
        features = []
        for angle in angles:
            features.extend([
                graycoprops(glcm, 'contrast')[0, 0],
                graycoprops(glcm, 'correlation')[0, 0],
                graycoprops(glcm, 'energy')[0, 0],
                graycoprops(glcm, 'homogeneity')[0, 0]
            ])
        columns = [f"{prop}_{int(np.degrees(angle))}" for prop in ('contrast', 'correlation', 'energy', 'homogeneity') for angle in angles]
        features_df = pd.DataFrame([features], columns=columns)
        features_df['Name'] = os.path.basename(image_path)
        
        features_df = features_df[['Name'] + [col for col in columns]]
        
        list_GLCM.append(features_df)
    glcm_df = pd.concat(list_GLCM, ignore_index=True)
    return glcm_df


In [9]:
def GLCM_all(image_paths, distance=3):
    list_GLCM = []
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    for image_path in tqdm(image_paths):
        try:
            from PIL import Image
            img = Image.open(image_path)
            img = img.convert("L")  
            img = np.array(img)
            
            if img.ndim != 2:
                if isinstance(img, list) or (img.ndim > 2 and img.shape[0] > 1):
                    img = img[0]
                else:
                    raise ValueError("Unexpected image shape")
            img = img.astype(np.uint8)
        except Exception as e:
            print(f"Failed to read image: {image_path}. Error: {e}")
            continue
        
        glcm = graycomatrix(img, [distance], angles, 256, symmetric=True, normed=True)
        features = []
        for angle in angles:
            features.extend([
                graycoprops(glcm, 'contrast')[0, 0],
                graycoprops(glcm, 'correlation')[0, 0],
                graycoprops(glcm, 'energy')[0, 0],
                graycoprops(glcm, 'homogeneity')[0, 0]
            ])
        columns = [f"{prop}_{int(np.degrees(angle))}" for prop in ('contrast', 'correlation', 'energy', 'homogeneity') for angle in angles]
        features_df = pd.DataFrame([features], columns=columns)
        features_df['Name'] = os.path.basename(image_path)
        features_df = features_df[['Name'] + [col for col in columns]]
        list_GLCM.append(features_df)
    glcm_df = pd.concat(list_GLCM, ignore_index=True)
    return glcm_df

SIFT features


In [10]:
sift = cv2.SIFT_create(nfeatures=800, contrastThreshold=0.02, edgeThreshold=10, sigma=1.2)

def extract_sift_features(image_paths, output_csv="sift_features.csv"):
    data = []

    for image_path in tqdm(image_paths):
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is not None:
            keypoints, descriptors = sift.detectAndCompute(image, None)
            if descriptors is not None:
                descriptor_mean = descriptors.mean(axis=0) 
            else:
                descriptor_mean = np.zeros(128)  

            image_name = os.path.basename(image_path)
            class_name = os.path.basename(os.path.dirname(image_path))  

            row = [image_name] + descriptor_mean.tolist() + [class_name]
            data.append(row)
        else:
            print(f"Failed to read {image_path}")

    column_names = ["Name"] + [f"sift_{i+1}" for i in range(128)] + ["class"]
    sift_df = pd.DataFrame(data, columns=column_names)

    sift_df.to_csv(output_csv, index=False)
    print(f"SIFT features saved to {output_csv}")

    return sift_df

In [11]:
# sift_features_df = extract_sift_features(image_paths)

In [32]:
# folder_path = r'D:\PROJECTWORSHOP\Soybean_Seeds\Pistachio_Image_Dataset\Kirmizi_Pistachio'
folder_path = r'D:\PROJECTWORSHOP\Canabis seeds\Dataset of Cannabis Seeds\Hang Kra Rog Phu Phan ST1'
image_paths = read_image_data(folder_path)

# basic_features_df = extract_basic_features(image_paths)
# basic_advanced_df = extract_advanced_features(image_paths)
# basic_advanced_df_CoHist = extract_advanced_features_CoHist(image_paths)
color_histogram_1D_df = color_histogram_1D(image_paths)
# color_histogram_3D_df = color_histogram_3D(image_paths)
# lbp_features_df = extract_LBP_features(image_paths)
# gist_features_df = extract_gist_features(image_paths)
# glcm_features_df = GLCM_all(image_paths)

The names of the first three images in the directory are: IMG_0555.JPG, IMG_0556.JPG, IMG_0557.JPG


100%|██████████| 249/249 [02:16<00:00,  1.82it/s]


In [ ]:
color_histogram_3D_df.shape

In [33]:
color_histogram_1D_df['class'] = 'Hang Kra Rog Phu Phan ST1'

color_histogram_1D_df.to_csv("Hang Kra Rog Phu Phan ST1.csv", index = False)

In [ ]:
print(f"{basic_features_df.shape}",)
print(f"{lbp_features_df.shape}",)
print(f"{gist_features_df.shape}",)
print(f"{glcm_features_df.shape}",)

(327, 19)
(327, 11)
(327, 129)
(327, 17)


In [ ]:
final_df1 = pd.merge(basic_features_df, gist_features_df, on='Name', how='inner')
final_df2 = pd.merge(final_df1, lbp_features_df, on='Name', how='inner')
final_df = pd.merge(final_df2, glcm_features_df)

# final_df = pd.DataFrame(basic_features_df)
final_df['class'] = 'Gelato_photo'

csv_file_path = 'Gelato_photo.csv'

final_df.to_csv(csv_file_path, index=False)

In [ ]:
final_df.shape

(203, 174)

In [34]:
df1 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\AK47_photo.csv')
df2 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\blackberry_auto.csv')
df3 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\cherry_pie.csv')
df4 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Gelato_photo.csv')
df5 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\gorillar_purple.csv')
df6 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Hang Kra Rog  KU.csv')
df7 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Hang Kra Rog Phu Phan ST1.csv')
df8 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Hang Suea Sakon Nakhon TT1.csv')
df9 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\KD.csv')
df10 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\KD_KT.csv')
df11 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Krerng Ka Via.csv')
df12 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\purple_duck.csv')
df13 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\skunk_auto.csv')
df14 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\sour_deisel_auto.csv')
df15 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Tanaosri Kan Daeng RD1.csv')
df16 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Tanaosri Kan Kaw WA1.csv')
df17 = pd.read_csv(r'D:\PROJECTWORSHOP\Canabis seeds\csv_data\Color Hist\Thaistick Foi Thong.csv')

In [35]:
print("df1 shape:", df1.shape)
print("df2 shape:", df2.shape)
print("df3 shape:", df3.shape)
print("df4 shape:", df4.shape)
print("df5 shape:", df5.shape)
print("df6 shape:", df6.shape)
print("df7 shape:", df7.shape)
print("df8 shape:", df8.shape)
print("df9 shape:", df9.shape)
print("df10 shape:", df10.shape)
print("df11 shape:", df11.shape)
print("df12 shape:", df12.shape)
print("df13 shape:", df13.shape)
print("df14 shape:", df14.shape)
print("df15 shape:", df15.shape)
print("df16 shape:", df16.shape)
print("df17 shape:", df17.shape)

df1 shape: (106, 26)
df2 shape: (203, 26)
df3 shape: (50, 26)
df4 shape: (327, 26)
df5 shape: (554, 26)
df6 shape: (153, 26)
df7 shape: (249, 26)
df8 shape: (93, 26)
df9 shape: (49, 26)
df10 shape: (147, 26)
df11 shape: (141, 26)
df12 shape: (151, 26)
df13 shape: (233, 26)
df14 shape: (327, 26)
df15 shape: (157, 26)
df16 shape: (183, 26)
df17 shape: (212, 26)


In [36]:
# Nối các DataFrame và đánh lại chỉ số
df_all = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13, df14, df15, df16,df17], ignore_index=True)

# Kiểm tra shape của DataFrame sau khi nối
print("Shape của DataFrame tổng hợp:", df_all.shape)


Shape của DataFrame tổng hợp: (3335, 26)


In [37]:
df_all.to_csv('color_histogram_features.csv', index=False)

In [ ]:
from PIL import Image

image_path = "D:\PROJECTWORSHOP\Canabis seeds\Dataset of Cannabis Seeds\cherry_pie\IMG_2278.JPG" 
img = Image.open(image_path)

width, height = img.size
print(f"Kích thước ảnh: {width} x {height} pixels")

dpi = img.info.get("dpi", "Không có thông tin DPI")
print(f"Độ phân giải: {dpi}")

Kích thước ảnh: 4032 x 3024 pixels
Độ phân giải: (300, 300)
